# Step 7: Model Evaluation

Evaluate model and generate plots

## Imports

In [ ]:
import time
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.metrics import (
    precision_recall_curve,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report
)

## Configuration

In [ ]:
# Configuration parameters
data_dir = "data"
output_dir = "outputs"
force = False  # Set to True to force rerun

## Setup and Initialization

In [ ]:
start_time = time.time()
print("=" * 60)
print("Step 7: Model Evaluation")
print("=" * 60)

# Setup paths
output_dir = Path(output_dir)
features_dir = output_dir / "features"
models_dir = output_dir / "models"
plots_dir = output_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

sentinel = plots_dir / "07_eval.done"
if sentinel.exists() and not force:
    print(f"[OK] Evaluation already complete (found {sentinel})")
    print(f"  Use --force to re-run")
else:
    # Check that required files exist
    model_file = models_dir / "catboost_model.cbm"
    if not model_file.exists():
        raise FileNotFoundError(
            f"Model file not found: {model_file}\n"
            f"Please run step 6 (train_model.py) first."
        )

## Load Model

In [ ]:
# Load model
print(f"Loading model from {model_file}...")
model = CatBoostClassifier()
model.load_model(str(model_file))

## Load Validation Data

In [ ]:
# Load validation data
print("Loading validation data...")
val_df = pd.read_parquet(features_dir / "val.parquet")

with open(features_dir / "feature_columns.json", 'r') as f:
    feature_cols = json.load(f)

with open(features_dir / "cat_feature_indices.json", 'r') as f:
    cat_indices = json.load(f)

X_val = val_df[feature_cols].copy()
y_val = val_df['is_outlier']

# Handle NaN in categorical features for CatBoost
for idx in cat_indices:
    col = feature_cols[idx]
    X_val[col] = X_val[col].fillna('UNKNOWN').astype(str)

print(f"  Validation size: {len(y_val)}")
print(f"  Outliers: {y_val.sum()} ({100 * y_val.mean():.2f}%)")

## Generate Predictions

In [ ]:
# Generate predictions
print("\nGenerating predictions...")
y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

## 1. Feature Importance Plot

In [ ]:
# 1. Feature Importance Plot
print("\n1. Creating feature importance plot...")
feature_importance = model.get_feature_importance()
feature_names = feature_cols

# Create DataFrame of top 30 features
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(30)

# Color features by type
colors = []
for feat in importance_df['feature']:
    if 'LOT_ID' in feat or '__EQUIP' in feat or '__POSITION' in feat or 'lot_' in feat:
        colors.append('red')  # Lot/categorical features
    elif '__SPC' in feat:
        colors.append('green')  # SPC features
    else:
        colors.append('blue')  # Sensor features

plt.figure(figsize=(12, 10))
plt.barh(range(len(importance_df)), importance_df['importance'], color=colors)
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Importance')
plt.title('Top 30 Feature Importances\n(Red=Lot/Categorical, Blue=Sensor, Green=SPC)')
plt.tight_layout()
plt.savefig(plots_dir / "feature_importance.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"  [OK] Saved feature_importance.png")

## 2. Precision-Recall Curve

In [ ]:
# 2. Precision-Recall Curve
print("2. Creating precision-recall curve...")
precision, recall, pr_thresholds = precision_recall_curve(y_val, y_pred_proba)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, linewidth=2, label=f'PR curve (AUC = {pr_auc:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / "precision_recall_curve.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"  [OK] Saved precision_recall_curve.png (AUC-PR = {pr_auc:.3f})")

## 3. ROC Curve

In [ ]:
# 3. ROC Curve
print("3. Creating ROC curve...")
fpr, tpr, roc_thresholds = roc_curve(y_val, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / "roc_curve.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"  [OK] Saved roc_curve.png (AUC-ROC = {roc_auc:.3f})")

## 4. Confusion Matrix

In [ ]:
# 4. Confusion Matrix
print("4. Creating confusion matrix...")
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Non-outlier', 'Outlier'],
            yticklabels=['Non-outlier', 'Outlier'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (threshold = 0.5)')
plt.tight_layout()
plt.savefig(plots_dir / "confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"  [OK] Saved confusion_matrix.png")

## Print Classification Report

In [ ]:
# Print classification report
print("\n" + "=" * 60)
print("Classification Report:")
print("=" * 60)
print(classification_report(y_val, y_pred, target_names=['Non-outlier', 'Outlier']))

## Print Top Features

In [ ]:
# Print top features
print("=" * 60)
print("Top 20 Most Important Features:")
print("=" * 60)
top_20 = importance_df.head(20)
for idx, row in top_20.iterrows():
    print(f"  {row['feature']:<60} {row['importance']:>8.2f}")

## Save Evaluation Summary

In [ ]:
# Save evaluation summary
summary_file = output_dir / "evaluation_summary.txt"
print(f"\nSaving evaluation summary to {summary_file}...")

with open(summary_file, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("Model Evaluation Summary\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"Validation Set Size: {len(y_val)}\n")
    f.write(f"Outliers: {y_val.sum()} ({100 * y_val.mean():.2f}%)\n")
    f.write(f"Non-outliers: {(1-y_val).sum()} ({100 * (1-y_val).mean():.2f}%)\n\n")

    f.write(f"ROC AUC: {roc_auc:.4f}\n")
    f.write(f"PR AUC: {pr_auc:.4f}\n\n")

    f.write("Confusion Matrix (threshold = 0.5):\n")
    f.write(f"  True Negatives:  {cm[0, 0]}\n")
    f.write(f"  False Positives: {cm[0, 1]}\n")
    f.write(f"  False Negatives: {cm[1, 0]}\n")
    f.write(f"  True Positives:  {cm[1, 1]}\n\n")

    f.write("Classification Report:\n")
    f.write(classification_report(y_val, y_pred, target_names=['Non-outlier', 'Outlier']))

    f.write("\n" + "=" * 60 + "\n")
    f.write("Top 20 Most Important Features:\n")
    f.write("=" * 60 + "\n")
    for idx, row in top_20.iterrows():
        f.write(f"{row['feature']:<60} {row['importance']:>8.2f}\n")

print(f"  [OK] Saved evaluation_summary.txt")

# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] Evaluation complete in {elapsed:.2f}s")
print("=" * 60)